<a href="https://colab.research.google.com/github/kolaparthisaimeenakshi-crypto/ai-mentor-portfolia/blob/main/day9_studyAgent_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search ddgs

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [8]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

Register for TCS NQT 2026. Check eligibility, salary, exam pattern, and selection process. Open for 2021 to 2027 batches. Apply For TCS NQT 2026 Hiring before May 14th, 2026. Apply online for TCS Recruitment 2026. Latest TCS Smart Hiring, BPS jobs, IT jobs, and engineering vacancies for freshers and experienced candidates. TCS Hiring 2026 is live with massive job openings for freshers and experien


In [9]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_2434/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [10]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '187b41ee-d8dc-4793-aa00-2f0c0ad95d7c', 'type': 'tool_call'}]

[2] ToolMessage
    Content: Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS jobs 2026, and all new TCS vacancies 2026 across India. Find complete details on TCS freshers hiring, eligibility, roles, and how to apply for top IT openings. Apr 8, 2026 · Tata Consultancy Services (TCS) (BSE

[3] AIMessage
    Content: [{'type': 'text', 'text': "TCS has made 25,000 job offers to fresh graduates for FY2026-27. Additionally, the company plans aggressive campus hiring, aiming to onboard around 40,000 freshers annually. It's also worth noting that TCS reported a headcount reduction of 23,460 in FY26.", 'extras': {'sig


In [11]:
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    It will give you the status of whether the domain name that you are looking for is available or not. Doing this for multiple domain names over time can help you build a list of available domain names. Use our WHOIS lookup tool to check domain name availability or to discover the contact information 

[3] AIMessage
    [{'type': 'text', 'text': 'The URL you provided, https://this-domain-does-not-exist-12345.example.com/jd, does not exist. The search results indicate that querying such a domain should result in an NXDOMAIN error, meaning the domain name is not found. Therefore, there is no content to display from t


In [12]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [13]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))

aws, spring boot


In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))

**Score: 2/10**

**Rationale:** Highly transactional, lacks research into TCS Digital, and shows no genuine interest in the role or company mission beyond compensation.


In [15]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_2434/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [16]:
import json, pathlib
profiles = json.loads(pathlib.Path('../content/sample_data/student_profiles.json').read_text())

for i, p in enumerate(profiles):
    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]} — {p["branch"]} CGPA {p["cgpa"]} → {p["target_company"]}')
    print(f'{"="*70}')

    msg = (f"I am {p['name']}, B.Tech {p['branch']} CGPA {p['cgpa']}, "
           f"skills: {', '.join(p['skills'])}. Target: {p['target_company']}. "
           f"Plan 3 mock interview questions for me, score one of my sample answers, "
           f"and tell me what skills I need to add to be a strong fit.")

    result = agent.invoke({'messages': [('user', msg)]}, config={'recursion_limit': 10})

    for j, m in enumerate(result['messages']):
        print(f'\n  [{j}] {type(m).__name__}')
        if hasattr(m, 'content') and m.content:
            print(f'      {str(m.content)[:300]}')
        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(f'      → tool_call: {tc.get("name")}({tc.get("args")})')


Student 1: Ravi Kumar — CSE CGPA 8.2 → TCS Digital

  [0] HumanMessage
      I am Ravi Kumar, B.Tech CSE CGPA 8.2, skills: Python, Java, SQL, Git. Target: TCS Digital. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Ravi! It\'s great to know your background and target. Let\'s get you prepared for TCS Digital.\n\nHere are 3 mock interview questions:\n\n1.  Tell me about a challenging project you\'ve worked on, what your role was, and how you overcame obstacles.\n2.  Explain the d

Student 2: Sneha Reddy — ECE CGPA 7.6 → Cognizant

  [0] HumanMessage
      I am Sneha Reddy, B.Tech ECE CGPA 7.6, skills: C++, Python, MATLAB, Verilog. Target: Cognizant. Plan 3 mock interview questions for me, score one of my sample answers, and tell me what skills I need to add to be a strong fit.

  [1] AIMessage
      [{'type': 'text', 'text': 'Hello Sneha, great to he

In [17]:
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79f1aaaca5a0>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    [{'type': 'text', 'text': 'I was unable to fetch the job description from the URL provided. Please check the URL and try again.', 'extras': {'signature': 'CpwFAQw51sdjp9zSY5FoyuNMmsBf+Xe2FtELcYADDewB4fTOQcdJ/JGkppWKQrv6g4LnrDlHpf+pfOdmODu4R3xJxqPGDRAH8DnnogQ8u2dDOlJ1X0t26udo0ZdosuDUd6zw2RdLC+b/Lu9ny
